## tl;dr
Partial High records increase eligible five-drone High/Medium flights from 2 to 10. With first >=5 pp / >=10 s windows, Medium top rank persists in High 1/10 and Low 6/18 flights. These are local-window ranks, not whole-stage averages.


## Context & Methods
### Key Assumptions
Use each battery’s frozen stage boundaries only; no Bideal rate conversion. Select earliest consecutive drop-to-drop segments, keeping ordinary integer-SOC plateaus. Test 3/5/8 pp windows and the next non-overlapping window. Each flight counts once per stage comparison.


In [1]:
from pathlib import Path
import sys,json
import pandas as pd
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'battery_normalization.py').exists())
sys.path.insert(0,str(ROOT/'output_py'))
from audit_partial_stage_rankings import OUT,rank_windows,digest


## Data
Raw wind-tunnel coordination CSV files and frozen battery boundaries. Source identities and SHA256 values are in manifest.json. Window details are in windows.csv; one-flight/stage, five-drone rates are in raw_rates_wide.csv.


In [2]:
manifest=json.loads((OUT/'manifest.json').read_text())
for path,expected in manifest['input_hashes'].items():
    assert digest(ROOT/path)==expected
windows=pd.read_csv(OUT/'windows.csv')
print(windows[['experiment_id','drone_id','stage','soc_start','soc_end','duration_s','raw_rate']].head(10).to_string(index=False))


                     experiment_id drone_id  stage  soc_start  soc_end  duration_s  raw_rate
wind_tunnel_column_75_head_lv1_001  drone_1   high         99       93      11.980 30.050083
wind_tunnel_column_75_head_lv1_001  drone_1   high         99       93      11.980 30.050083
wind_tunnel_column_75_head_lv1_001  drone_1   high         99       91      16.029 29.945723
wind_tunnel_column_75_head_lv1_001  drone_1   high         99       93      11.980 30.050083
wind_tunnel_column_75_head_lv1_001  drone_1   high         93       88      12.083 24.828271
wind_tunnel_column_75_head_lv1_001  drone_1 medium         82       79      32.022  5.621135
wind_tunnel_column_75_head_lv1_001  drone_1 medium         82       77      47.071  6.373351
wind_tunnel_column_75_head_lv1_001  drone_1 medium         82       74      76.995  6.234171
wind_tunnel_column_75_head_lv1_001  drone_1 medium         82       77      47.071  6.373351
wind_tunnel_column_75_head_lv1_001  drone_1 medium         77       72

## Results
Recompute rankings and denominators from the stored observed window endpoints. The extraction implementation is audit_partial_stage_rankings.py; scan() returns fresh windows without writing source data.


In [3]:
ranked,comparisons,summary=rank_windows(windows)
saved=pd.read_csv(OUT/'summary.csv')
pd.testing.assert_frame_equal(summary.reset_index(drop=True),saved,check_dtype=False)
print(summary[(summary.method=='endpoint')&(summary.policy=='first')].to_string(index=False))
print(json.loads((OUT/'validation.json').read_text())['matched_first_second'])


 threshold_pp policy   method stage  flights  unique_leader_flights  same_fastest  within5  within10  near_ties_2pct
            3  first endpoint  high       10                      9             2        8         9               9
            3  first endpoint   low       18                     18             6        6         9               1
            5  first endpoint  high       10                      9             1        8         9               9
            5  first endpoint   low       18                     18             6        7        10               3
            8  first endpoint  high        9                      9             2        9         9               9
            8  first endpoint   low       18                     18            10       10        11               0
[{'stage': 'high', 'flights': 7, 'first': 0, 'second': 2}, {'stage': 'low', 'flights': 18, 'first': 6, 'second': 13}]


## Takeaways
No 100%-start requirement remains. Do not equate an arbitrary small subsection with a stable whole-stage slope: thresholds and segment position affect rankings. High often has near-ties. Keep SOC range, duration and uncertainty alongside rates. No source experiment, flight control or Bideal model was changed.
